# Fiche TP N° 02 : Feature Extraction and Embeddings

## Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, f1_score
from gensim.models import Word2Vec, FastText
import matplotlib.pyplot as plt

print('✅ All libraries imported successfully!')

## Partie A : Préparation de données

### Preprocessing Pipeline (depuis TP01)
Le prétraitement n'ayant pas été sauvegardé dans un CSV lors du TP01, on l'applique ici directement sur `spooky.csv` pour générer `spooky_lemmatized.csv`.

In [ ]:
import re, nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

for r in ['punkt', 'stopwords', 'wordnet', 'omw-1.4', 'punkt_tab']:
    nltk.download(r, quiet=True)

STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def remove_repetitions(text):
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

def handle_homoglyphs(text):
    for k, v in {'$':'s','0':'o','1':'i','3':'e','4':'a','5':'s','7':'t','@':'a'}.items():
        text = text.replace(k, v)
    return text

def normalize_special(text):
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    return re.sub(r'<.*?>', '', text)

def full_preprocess(text):
    text = normalize_special(text)
    text = handle_homoglyphs(text)
    text = remove_repetitions(text)
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Charger spooky.csv brut et appliquer le pipeline
df_raw = pd.read_csv('spooky.csv')   # ajuster le chemin si nécessaire
print(f'Raw dataset: {df_raw.shape[0]} lignes')
print('Application du pipeline de prétraitement (~1-2 min)...')

df_raw['cleaned_text']    = df_raw['text'].apply(
    lambda t: re.sub(r'[^a-z\s]', ' ', normalize_special(t).lower()))
df_raw['text_lemmatized'] = df_raw['text'].apply(full_preprocess)

df_raw.to_csv('spooky_lemmatized.csv', index=False)
print('✅ Sauvegardé dans spooky_lemmatized.csv')
df_raw[['id', 'author', 'text', 'cleaned_text', 'text_lemmatized']].head()

In [ ]:
# Chargement du dataset pretraite
df = pd.read_csv('spooky_lemmatized.csv')

print(f'Dimensions : {df.shape[0]} lignes x {df.shape[1]} colonnes')
print(df['author'].value_counts())

# Gestion des valeurs nulles et suppression des textes vides
df['text_lemmatized'] = df['text_lemmatized'].fillna('')
df = df[df['text_lemmatized'].str.strip() != '']
df = df.reset_index(drop=True)

print(f'\nDataset propre : {df.shape[0]} lignes restantes')
df.head(5)

## Partie B : Encodage de la variable à prédire

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label'] = le.fit_transform(df['author'])

print('Correspondance encodage :')
for i, classe in enumerate(le.classes_):
    print(f'  {classe} -> {i}')

print(f'\nDistribution des labels encodes :')
print(df['label'].value_counts().sort_index())

print('\nAperçu du dataset avec labels :')
print(df[['author', 'label', 'text_lemmatized']].head())

## Partie C : Construction des bases d'entraînement et de test

In [ ]:
from sklearn.model_selection import train_test_split

X = df['text_lemmatized']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=0, stratify=y
)

print('='*57)
print('C.1 SPLIT TRAIN / TEST')
print('='*57)
print(f'Taille totale : {len(df)}')
print(f'Taille Train  : {len(X_train)} ({len(X_train)/len(df)*100:.1f}%)')
print(f'Taille Test   : {len(X_test)}  ({len(X_test)/len(df)*100:.1f}%)')

In [ ]:
print('='*57)
print('C.2 VÉRIFICATION DE LA STRATIFICATION')
print('='*57)

print('\nDistribution dans TRAIN :')
for label, count in pd.Series(y_train).value_counts().sort_index().items():
    print(f'  {le.classes_[label]} : {count} ({count/len(y_train)*100:.1f}%)')

print('\nDistribution dans TEST :')
for label, count in pd.Series(y_test).value_counts().sort_index().items():
    print(f'  {le.classes_[label]} : {count} ({count/len(y_test)*100:.1f}%)')

## Partie D : Méthodes de vectorisation

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# D.1 — Fréquence lexicale
cv = CountVectorizer()
X_train_cv = cv.fit_transform(X_train)
X_test_cv  = cv.transform(X_test)

print('='*57)
print('D.1 FRÉQUENCE LEXICALE — CountVectorizer')
print('='*57)
print(f'Taille du vocabulaire : {len(cv.get_feature_names_out())} mots')
print(f'Matrice Train         : {X_train_cv.shape}')
print(f'Matrice Test          : {X_test_cv.shape}')
print('\nExtrait matrice Train (5 docs x premiers mots) :')
print(pd.DataFrame.sparse.from_spmatrix(
    X_train_cv[:5], columns=cv.get_feature_names_out()).iloc[:, :8])

In [ ]:
# D.2 — One-Hot Encoding
cv_ohe = CountVectorizer(binary=True)
X_train_ohe = cv_ohe.fit_transform(X_train)
X_test_ohe  = cv_ohe.transform(X_test)

print('='*57)
print('D.2 ONE-HOT ENCODING — CountVectorizer binaire')
print('='*57)
print(f'Taille du vocabulaire : {len(cv_ohe.get_feature_names_out())} mots')
print(f'Matrice Train         : {X_train_ohe.shape}')
print(f'Matrice Test          : {X_test_ohe.shape}')
print(f'Valeur max (doit etre 1) : {X_train_ohe.max()}')

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# D.3 — TF-IDF
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

idf_scores = pd.Series(tfidf.idf_, index=tfidf.get_feature_names_out())
print('Top 10 mots avec IDF le plus élevé (mots rares/discriminants) :')
print(idf_scores.sort_values(ascending=False).head(10))
print(f'\nMatrice TF-IDF Train : {X_train_tfidf.shape}')
print(f'Matrice TF-IDF Test  : {X_test_tfidf.shape}')

## Partie E : Entraînement et Évaluation des Modèles

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
import time

nom_classes = le.classes_

# Création des modèles
mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=0)
lr  = LogisticRegression(max_iter=1000, random_state=0)
nb  = MultinomialNB()

# Chaque modèle est associé à UNE représentation (comme dans le rapport)
modeles = {
    'MLP'                 : mlp,
    'Logistic Regression' : lr,
    'Naive Bayes'         : nb
}

representations_train = {
    'Fréquence Lexicale' : X_train_cv,
    'One-Hot Encoding'   : X_train_ohe,
    'TF-IDF'             : X_train_tfidf
}

representations_test = {
    'Fréquence Lexicale' : X_test_cv,
    'One-Hot Encoding'   : X_test_ohe,
    'TF-IDF'             : X_test_tfidf
}

print('Modèles définis ✅')

In [ ]:
# Entrainement croisé + mesure du temps
temps_entrainement = {}

for (nom_modele, modele), (nom_repr, X_tr) in zip(modeles.items(), representations_train.items()):
    t0 = time.time()
    modele.fit(X_tr, y_train)
    temps_entrainement[f'{nom_modele}_{nom_repr}'] = time.time() - t0
    print(f'Modele {nom_modele} entrainé sur {nom_repr}')

In [ ]:
# E.3 & E.4 — Prédictions et Rapports sur les données d'ENTRAÎNEMENT
print('='*57)
print('E.3 PRÉDICTIONS SUR LES DONNÉES D\'ENTRAÎNEMENT')
print('='*57)

for (nom_modele, modele), (nom_repr, X_tr) in zip(modeles.items(), representations_train.items()):
    y_pred = modele.predict(X_tr)
    acc = accuracy_score(y_train, y_pred)
    print(f'  {nom_modele} / {nom_repr} -> Accuracy Train : {acc*100:.2f}%')

print('\n')
for (nom_modele, modele), (nom_repr, X_tr) in zip(modeles.items(), representations_train.items()):
    y_pred = modele.predict(X_tr)
    print(f'Modele : {nom_modele} | Représentation : {nom_repr}')
    print(classification_report(y_train, y_pred, target_names=nom_classes))

## Partie F : Évaluation sur les données de Test

In [ ]:
# F.1 : Predictions sur les donnees de test
predictions_test = {}

print('='*57)
print('F.1 PRÉDICTIONS SUR LES DONNÉES DE TEST')
print('='*57)

for (nom_modele, modele), (nom_repr, X_te) in zip(modeles.items(), representations_test.items()):
    y_pred = modele.predict(X_te)
    predictions_test[f'{nom_modele}_{nom_repr}'] = y_pred
    acc = accuracy_score(y_test, y_pred)
    print(f'  {nom_modele} / {nom_repr} -> Accuracy Test : {acc*100:.2f}%')

In [ ]:
# F.2 : Rapports detailles
for (nom_modele, modele), (nom_repr, X_te) in zip(modeles.items(), representations_test.items()):
    cle    = f'{nom_modele}_{nom_repr}'
    y_pred = predictions_test[cle]
    print(f'Modele : {nom_modele} | Representation : {nom_repr}')
    print(classification_report(y_test, y_pred, target_names=nom_classes))

In [ ]:
# F.3 : Tableau recapitulatif final avec temps d'entraînement
print('='*57)
print('F.3 TEMPS D\'ENTRAÎNEMENT + TABLEAU RÉCAPITULATIF FINAL')
print('='*57)

tableau = []
for (nom_modele, modele), (nom_repr, X_te) in zip(modeles.items(), representations_test.items()):
    cle    = f'{nom_modele}_{nom_repr}'
    y_pred = predictions_test[cle]
    temps  = temps_entrainement[cle]
    tableau.append({
        'Modele'           : nom_modele,
        'Representation'   : nom_repr,
        'Accuracy'         : f'{accuracy_score(y_test, y_pred)*100:.2f}%',
        'Precision (macro)': f'{f1_score(y_test, y_pred, average="macro"):.3f}',
        'Recall (macro)'   : f'{f1_score(y_test, y_pred, average="macro"):.3f}',
        'F1 (macro)'       : f'{f1_score(y_test, y_pred, average="macro"):.3f}',
        'Temps (s)'        : f'{temps:.3f}'
    })

df_recap = pd.DataFrame(tableau)
print(df_recap.to_string(index=False))

## Partie G : Vectorisations basées sur les Word Embeddings

In [ ]:
# Tokenisation pour Gensim
X_train_tok = [text.split() for text in X_train]
X_test_tok  = [text.split() for text in X_test]

VECTOR_SIZE = 100
EPOCHS      = 10
WINDOW      = 5
MIN_COUNT   = 2

def document_vector(model, tokens, vector_size):
    vecs = []
    for word in tokens:
        try:
            vecs.append(model.wv[word])
        except KeyError:
            pass
    return np.mean(vecs, axis=0) if vecs else np.zeros(vector_size)

def corpus_to_matrix(model, tokenized_corpus, vector_size):
    return np.array([document_vector(model, t, vector_size) for t in tokenized_corpus])

print(f'Corpus tokenisé. Exemple : {X_train_tok[0][:8]} ...')

In [ ]:
# G.1a — Word2Vec CBOW (sg=0)
from gensim.models import Word2Vec

w2v_cbow = Word2Vec(sentences=X_train_tok, vector_size=VECTOR_SIZE,
                    window=WINDOW, min_count=MIN_COUNT, sg=0,
                    epochs=EPOCHS, workers=4, seed=42)

X_train_cbow = corpus_to_matrix(w2v_cbow, X_train_tok, VECTOR_SIZE)
X_test_cbow  = corpus_to_matrix(w2v_cbow, X_test_tok,  VECTOR_SIZE)
print('W2V-CBOW — Train shape:', X_train_cbow.shape)

In [ ]:
# G.1b — Word2Vec Skip-Gram (sg=1)
w2v_sg = Word2Vec(sentences=X_train_tok, vector_size=VECTOR_SIZE,
                  window=WINDOW, min_count=MIN_COUNT, sg=1,
                  epochs=EPOCHS, workers=4, seed=42)

X_train_sg = corpus_to_matrix(w2v_sg, X_train_tok, VECTOR_SIZE)
X_test_sg  = corpus_to_matrix(w2v_sg, X_test_tok,  VECTOR_SIZE)
print('W2V-SkipGram — Train shape:', X_train_sg.shape)

In [ ]:
# G.2 — GloVe
# Option A : GloVe pré-entraîné (décommenter si glove.6B.100d.txt est disponible)
# from gensim.scripts.glove2word2vec import glove2word2vec
# from gensim.models import KeyedVectors
# glove2word2vec('glove.6B.100d.txt', 'glove.word2vec.txt')
# glove_kv = KeyedVectors.load_word2vec_format('glove.word2vec.txt', binary=False)
# def dv_kv(kv, tokens, vs):
#     vecs = [kv[w] for w in tokens if w in kv]
#     return np.mean(vecs, axis=0) if vecs else np.zeros(vs)
# X_train_glove = np.array([dv_kv(glove_kv, t, VECTOR_SIZE) for t in X_train_tok])
# X_test_glove  = np.array([dv_kv(glove_kv, t, VECTOR_SIZE) for t in X_test_tok])

# Option B : Approximation sur corpus (grande fenêtre de contexte)
glove_approx = Word2Vec(sentences=X_train_tok, vector_size=VECTOR_SIZE,
                        window=10, min_count=MIN_COUNT, sg=0,
                        epochs=EPOCHS, workers=4, seed=42)

X_train_glove = corpus_to_matrix(glove_approx, X_train_tok, VECTOR_SIZE)
X_test_glove  = corpus_to_matrix(glove_approx, X_test_tok,  VECTOR_SIZE)
print('GloVe (approx) — Train shape:', X_train_glove.shape)

In [ ]:
# G.3 — FastText
from gensim.models import FastText

ft_model = FastText(sentences=X_train_tok, vector_size=VECTOR_SIZE,
                    window=WINDOW, min_count=MIN_COUNT, sg=1,
                    epochs=EPOCHS, workers=4, seed=42)

X_train_ft = np.array([
    np.mean([ft_model.wv[w] for w in t], axis=0) if t else np.zeros(VECTOR_SIZE)
    for t in X_train_tok])
X_test_ft  = np.array([
    np.mean([ft_model.wv[w] for w in t], axis=0) if t else np.zeros(VECTOR_SIZE)
    for t in X_test_tok])
print('FastText — Train shape:', X_train_ft.shape)

## Partie H : Entraînement / Test — Word Embeddings

In [ ]:
embed_representations = {
    'W2V-CBOW'    : (X_train_cbow,  X_test_cbow),
    'W2V-SkipGram': (X_train_sg,    X_test_sg),
    'GloVe'       : (X_train_glove, X_test_glove),
    'FastText'    : (X_train_ft,    X_test_ft),
}

embed_results = {}

for nom_repr, (X_tr, X_te) in embed_representations.items():
    clf = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=0)
    t0  = time.time()
    clf.fit(X_tr, y_train)
    t_train = time.time() - t0

    y_pred_tr = clf.predict(X_tr)
    y_pred_te = clf.predict(X_te)

    print(f'\n{"="*55}')
    print(f'  MLP + {nom_repr}   (temps : {t_train:.2f}s)')
    print(f'{"="*55}')
    print('  --- Train ---')
    print(classification_report(y_train, y_pred_tr, target_names=nom_classes))
    print('  --- Test ---')
    print(classification_report(y_test, y_pred_te, target_names=nom_classes))

    embed_results[nom_repr] = {
        'test_acc'   : accuracy_score(y_test, y_pred_te),
        'f1_macro'   : f1_score(y_test, y_pred_te, average='macro'),
        'train_time' : t_train
    }

## H.2 — Comparaison de tous les modèles

In [ ]:
all_results = []

# Résultats BoW / OHE / TF-IDF
for (nom_modele, modele), (nom_repr, X_te) in zip(modeles.items(), representations_test.items()):
    cle    = f'{nom_modele}_{nom_repr}'
    y_pred = predictions_test[cle]
    all_results.append({
        'Modèle'        : nom_modele,
        'Vectorisation' : nom_repr,
        'Test Accuracy' : round(accuracy_score(y_test, y_pred), 4),
        'F1 macro'      : round(f1_score(y_test, y_pred, average='macro'), 4),
        'Temps (s)'     : round(temps_entrainement[cle], 3)
    })

# Résultats Embeddings
for nom_repr, res in embed_results.items():
    all_results.append({
        'Modèle'        : 'MLP',
        'Vectorisation' : nom_repr,
        'Test Accuracy' : round(res['test_acc'], 4),
        'F1 macro'      : round(res['f1_macro'], 4),
        'Temps (s)'     : round(res['train_time'], 3)
    })

df_all = pd.DataFrame(all_results).sort_values('Test Accuracy', ascending=False).reset_index(drop=True)
print('\n===== COMPARAISON COMPLÈTE DE TOUS LES MODÈLES =====\n')
print(df_all.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

labels = [f"{r['Modèle']}\n{r['Vectorisation']}" for _, r in df_all.iterrows()]

axes[0].barh(labels, df_all['Test Accuracy'], color='steelblue')
axes[0].set_xlabel('Test Accuracy')
axes[0].set_title('Accuracy par méthode')
axes[0].set_xlim([0, 1])
for i, v in enumerate(df_all['Test Accuracy']):
    axes[0].text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=8)

axes[1].barh(labels, df_all['Temps (s)'], color='darkorange')
axes[1].set_xlabel('Temps d\'entraînement (s)')
axes[1].set_title('Temps d\'entraînement par méthode')
for i, v in enumerate(df_all['Temps (s)']):
    axes[1].text(v + 0.1, i, f'{v:.1f}s', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('comparison_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## Conclusion

Ce TP nous a permis de voir l'importance cruciale de la vectorisation dans le NLP. Des modèles simples comme **Naive Bayes** sont très rapides et efficaces sur les données textuelles, tandis que le **MLP** offre des résultats plus puissants au prix d'un temps de calcul plus élevé. La méthode **TF-IDF** s'est avérée particulièrement performante pour isoler les caractéristiques propres à chaque auteur.

| Méthode | Avantages | Inconvénients |
|---|---|---|
| **BoW (CountVectorizer)** | Simple, rapide | Ignore l'ordre, vecteurs creux |
| **One-Hot Encoding** | Simple | Ignore la fréquence |
| **TF-IDF** | Pondère l'importance des mots | Ignore le contexte |
| **Word2Vec CBOW** | Capture le contexte local | Ignore les sous-mots |
| **Word2Vec Skip-Gram** | Mieux pour mots rares | Plus lent que CBOW |
| **GloVe** | Statistiques globales + locales | Nécessite grands corpus |
| **FastText** | Gère les mots inconnus (sous-mots) | Plus gourmand en mémoire |
